# 회계/재무 손익계산서 데모 (Management-ToolKit)

이 노트북은 [Management-ToolKit](https://github.com/Dubong-hedgehog/Management-ToolKit) 저장소의
`finance/income_statement_generator.py` 로직을 그대로 가져와서, 설치 없이 브라우저에서 바로
실행해볼 수 있게 만든 데모입니다.

- 위에서부터 순서대로 셀을 실행하세요 (각 셀 왼쪽 ▶ 버튼, 또는 Shift+Enter)
- 기본값은 저장소의 **가짜 샘플 데이터**입니다
- 원하면 직접 CSV를 업로드해서 다른 숫자로도 돌려볼 수 있습니다 (아래 '내 데이터 업로드' 섹션, 선택사항)


In [ ]:
import os
print("skip clone, using local copy")


In [ ]:
import sys
import pandas as pd
sys.path.insert(0, ".")

from common.excel_io import load_csv
from common.period_utils import add_period_label, available_periods
from finance.income_statement_generator import (
    build_comparison_table, print_statement, plot_trend,
    REVENUE_ACCOUNTS, COGS_ACCOUNTS, OPEX_ACCOUNTS,
    NON_OP_INCOME_ACCOUNTS, NON_OP_EXPENSE_ACCOUNTS,
)

print("불러오기 완료. 기본 계정과목 매핑:")
print("매출:", REVENUE_ACCOUNTS)
print("매출원가:", COGS_ACCOUNTS)
print("판관비:", OPEX_ACCOUNTS)
print("영업외수익:", NON_OP_INCOME_ACCOUNTS)
print("영업외비용:", NON_OP_EXPENSE_ACCOUNTS)


## 1. 데이터 준비

기본값은 저장소의 샘플 데이터(`finance/sample_data/sample_transactions.csv`)입니다.
바로 다음 섹션(2번)으로 넘어가서 조회해봐도 되고, 내 데이터로 해보고 싶다면 아래
'내 데이터 업로드' 셀을 실행하세요.

In [ ]:
df = load_csv("finance/sample_data/sample_transactions.csv")
print(f"샘플 데이터 {len(df)}건 로드 완료 (기간: {df['거래일자'].min()} ~ {df['거래일자'].max()})")
df.head()


### (선택) 내 데이터로 해보고 싶다면

`거래일자, 계정과목, 금액` 세 컬럼을 가진 CSV 파일을 업로드하세요.
계정과목 이름이 위 매핑(매출/매출원가/급여 등)과 다르면, 업로드 후 나오는 안내에 따라
그 다음 셀에서 계정과목 매핑을 내 데이터에 맞게 바꿔주세요.

In [ ]:
try:
    from google.colab import files
    import io
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[fname]), encoding="utf-8-sig")
        print(f"'{fname}' 업로드 완료: {len(df)}건")
        known = REVENUE_ACCOUNTS | COGS_ACCOUNTS | OPEX_ACCOUNTS | NON_OP_INCOME_ACCOUNTS | NON_OP_EXPENSE_ACCOUNTS
        missing = set(df["계정과목"].unique()) - known
        if missing:
            print(f"\n[알림] 아래 계정과목은 매핑에 없어 집계에서 빠집니다: {missing}")
            print("바로 아래 셀에서 REVENUE_ACCOUNTS 등에 추가해주세요.")
except ImportError:
    print("Colab 환경이 아니라 업로드 기능을 쓸 수 없습니다. 샘플 데이터를 계속 사용합니다.")


In [ ]:
# 필요하면 내 회사 계정과목명으로 아래 주석을 풀어서 수정하세요 (예시)
# REVENUE_ACCOUNTS = {"매출", "용역매출"}
# COGS_ACCOUNTS = {"매출원가"}
# OPEX_ACCOUNTS = {"급여", "임차료", "복리후생비"}
# NON_OP_INCOME_ACCOUNTS = {"이자수익"}
# NON_OP_EXPENSE_ACCOUNTS = {"이자비용"}


## 2. 기간 선택해서 손익계산서 보기

기간 단위(년/반기/분기/월/주)를 고르고 조회할 기간을 선택한 뒤 '조회' 버튼을 누르면,
K-IFRS 형식 손익계산서(직전기·전년동기 비교 포함)와 추이 차트가 바로 아래에 나타납니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

period_type_dd = widgets.Dropdown(options=["year", "half", "quarter", "month", "week"], value="month", description="기간단위:")
period_dd = widgets.Dropdown(description="기간:")
run_btn = widgets.Button(description="조회", button_style="primary")
out = widgets.Output()
state = {"labeled": None}

def refresh_periods(*_):
    labeled = add_period_label(df, "거래일자", period_type_dd.value)
    state["labeled"] = labeled
    opts = available_periods(labeled, "기간")
    period_dd.options = opts
    if opts:
        period_dd.value = opts[-1]

def on_run_clicked(_):
    with out:
        clear_output(wait=True)
        labeled = state["labeled"]
        table = build_comparison_table(labeled, "기간", period_dd.value, period_type_dd.value)
        print_statement(table, period_dd.value)
        chart_path = plot_trend(labeled, "기간", period_type_dd.value, "/tmp/colab_trend.png")
        display(Image(str(chart_path)))

period_type_dd.observe(refresh_periods, names="value")
run_btn.on_click(on_run_clicked)
refresh_periods()

display(widgets.HBox([period_type_dd, period_dd, run_btn]))
display(out)


---
법인세비용은 이 기간 손익을 연 환산해 국세청 세율 구간(지방소득세 포함)에 대입한
**추정치**입니다. 세무조정, 이월결손금 등은 반영되지 않아 실제 신고세액과 다를 수
있습니다. 계산 로직은 저장소의 `common/tax_utils.py`, `common/period_utils.py`를
참고하세요.